# Home Value Estimator — Model Notebook

Goal: clean a messy synthetic housing dataset, explore it, train and compare
three regression models, and export the best pipeline for the Flask API.

Pipeline: **raw CSV → cleaning → EDA → feature engineering → train/compare → export**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, joblib, warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42

## 1. Load raw data

In [ ]:
df_raw = pd.read_csv("data/homes.csv")
print(df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

## 2. Cleaning

The raw file has two deliberate messiness patterns to handle:

1. `sale_price` mixes `"$287.7K"` style strings with plain numbers.
2. `sqft_living` mixes square feet and square meters, flagged by `area_unit`.

Plus ordinary missing values in a handful of columns.

In [ ]:
def parse_price(val):
    if isinstance(val, str) and val.strip().upper().endswith("K"):
        return float(val.strip().replace("$", "").replace("K", "")) * 1000
    return float(val)

df = df_raw.copy()
df["sale_price"] = df["sale_price"].apply(parse_price)

# normalize every area to square feet
sqm_mask = df["area_unit"].eq("sqm")
df.loc[sqm_mask, "sqft_living"] = df.loc[sqm_mask, "sqft_living"] * 10.7639
df = df.drop(columns=["area_unit"])

df[["sale_price", "sqft_living"]].describe()

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
# year_built: fill with neighborhood median (structural, not random)
df["year_built"] = df.groupby("neighborhood")["year_built"].transform(
    lambda s: s.fillna(s.median())
)
df["age"] = 2024 - df["year_built"]

# school_score: fill with overall median, it's a soft signal not tied to neighborhood alone
df["school_score"] = df["school_score"].fillna(df["school_score"].median())

# basement / lot_size: missing at random, small fraction -> mode / median
df["basement"] = df["basement"].fillna(df["basement"].mode()[0])
df["lot_size_sqft"] = df["lot_size_sqft"].fillna(df["lot_size_sqft"].median())

assert df.isna().sum().sum() == 0, "still missing values"
df.shape

## 3. Outlier handling

Trim extreme `sale_price` and `sqft_living` values using the IQR rule so a
handful of mansions/typos don't dominate the fit.

In [ ]:
def iqr_bounds(s, k=3.0):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

lo_p, hi_p = iqr_bounds(df["sale_price"])
lo_a, hi_a = iqr_bounds(df["sqft_living"])

before = len(df)
df = df[df["sale_price"].between(lo_p, hi_p) & df["sqft_living"].between(lo_a, hi_a)]
print(f"Dropped {before - len(df)} outlier rows ({(before-len(df))/before:.1%})")

## 4. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["sale_price"], bins=40, ax=axes[0], color="#3A6B69")
axes[0].set_title("Sale price distribution")
sns.histplot(np.log1p(df["sale_price"]), bins=40, ax=axes[1], color="#C9A227")
axes[1].set_title("log1p(sale price) distribution")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x="sqft_living", y="sale_price", hue="grade", palette="viridis", alpha=0.5, legend=False)
plt.title("Living area vs. sale price (color = grade)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
order = df.groupby("neighborhood")["sale_price"].median().sort_values().index
sns.boxplot(data=df, x="neighborhood", y="sale_price", order=order, palette="crest")
plt.xticks(rotation=30, ha="right")
plt.title("Sale price by neighborhood")
plt.tight_layout()
plt.show()

In [ ]:
num_cols = ["sale_price", "sqft_living", "lot_size_sqft", "bedrooms", "bathrooms",
            "floors", "age", "condition", "grade", "school_score"]
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

## 5. Feature engineering

Nothing fancy — a couple of ratios that a human appraiser would also reach
for, plus casting the yes/no columns to booleans.

In [ ]:
df["price_per_sqft_hint"] = df["sale_price"] / df["sqft_living"]  # inspection only, dropped before training
df["bath_bed_ratio"] = df["bathrooms"] / df["bedrooms"].replace(0, 1)
df["lot_to_living_ratio"] = df["lot_size_sqft"] / df["sqft_living"]

for col in ["renovated", "garage", "basement", "pool"]:
    df[col] = df[col].map({"Yes": 1, "Y": 1, "No": 0, "N": 0}).astype(int)

df.drop(columns=["price_per_sqft_hint"]).head()

## 6. Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = ["neighborhood", "sqft_living", "lot_size_sqft", "bedrooms", "bathrooms",
            "floors", "age", "renovated", "condition", "grade", "garage",
            "basement", "pool", "school_score", "bath_bed_ratio", "lot_to_living_ratio"]
TARGET = "sale_price"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_train.shape, X_test.shape

## 7. Preprocessing + model pipelines

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

CAT_COLS = ["neighborhood"]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), NUM_COLS),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), CAT_COLS),
])

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

candidates = {
    "Ridge": Ridge(alpha=5.0, random_state=RANDOM_STATE),
    "RandomForest": RandomForestRegressor(
        n_estimators=120, max_depth=10, min_samples_leaf=4, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=120, max_depth=10, min_samples_leaf=4, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

results = []
fitted = {}

for name, model in candidates.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2 = r2_score(y_test, preds)
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2")

    results.append({"model": name, "MAE": mae, "RMSE": rmse, "R2": r2,
                     "CV_R2_mean": cv.mean(), "CV_R2_std": cv.std()})
    fitted[name] = pipe

results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
results_df

### Did a log-target transform help?

Worth checking rather than assuming — house prices are right-skewed, so a
`log1p` target sometimes helps linear models. Tested here on the best
candidate so far.

In [ ]:
from sklearn.compose import TransformedTargetRegressor

best_name = results_df.loc[0, "model"]
best_plain_r2 = results_df.loc[0, "R2"]

log_model = TransformedTargetRegressor(
    regressor=Pipeline([("prep", preprocess), ("model", candidates[best_name].__class__(**candidates[best_name].get_params()))]),
    func=np.log1p, inverse_func=np.expm1,
)
log_model.fit(X_train, y_train)
log_preds = log_model.predict(X_test)
log_r2 = r2_score(y_test, log_preds)

print(f"{best_name} plain R2:  {best_plain_r2:.4f}")
print(f"{best_name} log1p R2:  {log_r2:.4f}")
print("log1p helped" if log_r2 > best_plain_r2 else "log1p did NOT help — keeping the plain target")

## 8. Pick the winner and inspect feature importance

In [ ]:
winner_name = results_df.loc[0, "model"]
winner = fitted[winner_name]
print("Exporting:", winner_name)

feature_names = (
    NUM_COLS
    + list(winner.named_steps["prep"].named_transformers_["cat"].named_steps["ohe"].get_feature_names_out(CAT_COLS))
)

if hasattr(winner.named_steps["model"], "feature_importances_"):
    importances = winner.named_steps["model"].feature_importances_
    imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False).head(12)

    plt.figure(figsize=(7, 5))
    sns.barplot(data=imp_df, x="importance", y="feature", color="#3A6B69")
    plt.title(f"{winner_name} — top feature importances")
    plt.tight_layout()
    plt.show()
else:
    print(f"{winner_name} has no feature_importances_ (linear model — see coefficients instead).")

## 9. Export the pipeline + metrics for the API

In [ ]:
import os
os.makedirs("../backend/models", exist_ok=True)

joblib.dump(winner, "../backend/models/home_value_pipeline.pkl")

metrics = {
    "model_name": winner_name,
    "model_version": "0.1.0",
    "mae": float(results_df.loc[results_df.model == winner_name, "MAE"].iloc[0]),
    "rmse": float(results_df.loc[results_df.model == winner_name, "RMSE"].iloc[0]),
    "r2": float(results_df.loc[results_df.model == winner_name, "R2"].iloc[0]),
    "cv_r2_mean": float(results_df.loc[results_df.model == winner_name, "CV_R2_mean"].iloc[0]),
    "cv_r2_std": float(results_df.loc[results_df.model == winner_name, "CV_R2_std"].iloc[0]),
    "n_train_rows": int(len(X_train)),
    "n_test_rows": int(len(X_test)),
    "log_target_helped": bool(log_r2 > best_plain_r2),
}

with open("../backend/models/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

with open("../frontend/assets/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

neighborhoods = sorted(df["neighborhood"].unique().tolist())
with open("../frontend/assets/neighborhoods.json", "w") as f:
    json.dump(neighborhoods, f, indent=2)

metrics

## 10. Summary

- Cleaned mixed-format price and area fields, imputed missing values with
  column-appropriate strategies (not a blanket `fillna(0)`).
- Compared **Ridge**, **RandomForest**, and **ExtraTrees**; picked the
  highest test-set R².
- Checked whether a `log1p` price transform helps instead of assuming it
  does.
- Exported the winning `sklearn` pipeline (imputer → scaler/OHE → model) as
  one `.pkl`, plus a metrics JSON consumed by both the API and the frontend
  hero stats.